# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their fields by @id

print("Available record sets (@id):\n")
for record_set in metadata.record_sets:
    print(f"- RecordSet @id: {record_set.id} | name: {record_set.name}")
    print(f"  Fields:")
    for field in record_set.fields:
        print(f"   - Field @id: {field.id} | name: {field.name} | data type: {field.data_type}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
record_sets = [record_set.id for record_set in metadata.record_sets]
dataframes = {}

for record_set_id in record_sets:
    # Each record yields a dict mapping field @id to value
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} rows for RecordSet: {record_set_id}")
        print("  Columns:", df.columns.tolist())
    except Exception as e:
        print(f"Could not load RecordSet {record_set_id}: {e}")

# Choose a record set for subsequent operations
if record_sets:
    example_record_set_id = record_sets[0]
    print(f"\nColumns in {example_record_set_id}:")
    print(dataframes[example_record_set_id].columns.tolist())
    dataframes[example_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# -- Please edit these IDs depending on your field overview (from Section 2) --
# For demonstration, we select a numeric field and a grouping field
record_set_id = example_record_set_id  # From above
df = dataframes[record_set_id]

# Attempt to find a numeric field for demonstration purposes
possible_numeric_fields = [col for col in df.columns if df[col].dtype.kind in ["i", "f"]]
if possible_numeric_fields:
    numeric_field = possible_numeric_fields[0]  # Use the first one found
else:
    print("No numeric field found. Using first available column.")
    numeric_field = df.columns[0]

# Try to find a grouping field with a manageable number of unique categories
possible_group_fields = [col for col in df.columns if df[col].nunique() < 20 and col != numeric_field]
if possible_group_fields:
    group_field = possible_group_fields[0]
else:
    group_field = None

# Filtering records based on a threshold
threshold = df[numeric_field].quantile(0.75) if df[numeric_field].dtype.kind in ["i", "f"] else None
print(f"\nSelecting {numeric_field} with threshold > {threshold}\n")
filtered_df = df[df[numeric_field] > threshold] if threshold is not None else df
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalization
if df[numeric_field].dtype.kind in ["i", "f"]:
    mean = filtered_df[numeric_field].mean()
    std = filtered_df[numeric_field].std()
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - mean) / std
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Grouping
if group_field is not None and group_field in df.columns:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"\nGrouped data by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

# Histogram of the selected numeric field
if numeric_field and df[numeric_field].dtype.kind in ['i', 'f']:
    plt.figure(figsize=(8, 5))
    df[numeric_field].hist(bins=30, color='steelblue', edgecolor='black')
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

# Boxplot grouped by group_field if available
if group_field is not None and df[group_field].nunique() < 10 and df[numeric_field].dtype.kind in ['i', 'f']:
    plt.figure(figsize=(10,5))
    df.boxplot(column=numeric_field, by=group_field)
    plt.title(f"{numeric_field} by {group_field}")
    plt.suptitle("")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Successfully loaded metadata, record set structure, and data using `mlcroissant` referencing all entities by their `@id`.
- Explored the available fields and performed filtering, normalization, and aggregation operations directly using `@id` field references.
- Visualized primary numeric field distributions, which can inform further mass spectrometry-based biological research and downstream analysis.